# 01. 기초: keyed green list와 z-score

목표: secret key와 prefix가 후보 token을 어떻게 나누고, 작은 logit bias가 긴 sequence에서 어떻게 검출 가능한 count 차이가 되는지 확인한다. 이 toy 구현은 실제 provider watermark가 아니다.

In [ ]:
import hashlib
import hmac
import math
import random

VOCAB = ['important', 'significant', 'substantial', 'notable', 'clear', 'useful', 'robust', 'simple']
BASE_LOGITS = [1.4, 1.1, 0.8, 0.4, 1.0, 0.7, 0.5, 0.3]
GAMMA = 0.5

def green_set(key: bytes, prefix: tuple[str, ...], gamma: float = GAMMA) -> set[str]:
    # HMAC 점수가 낮은 token 절반을 green으로 정한다. key와 prefix가 같으면 완전히 재현된다.
    ranked = []
    context = ' '.join(prefix).encode()
    for token in VOCAB:
        score = hmac.new(key, context + b'|' + token.encode(), hashlib.sha256).digest()
        ranked.append((score, token))
    ranked.sort()
    return {token for _, token in ranked[: round(len(VOCAB) * gamma)]}

def softmax(logits: list[float]) -> list[float]:
    top = max(logits)
    weights = [math.exp(value - top) for value in logits]
    total = sum(weights)
    return [value / total for value in weights]

prefix = ('the', 'result', 'was')
greens = green_set(b'classroom-key', prefix)
base = softmax(BASE_LOGITS)
biased = softmax([logit + (1.2 if token in greens else 0.0) for token, logit in zip(VOCAB, BASE_LOGITS)])
for token, p0, p1 in zip(VOCAB, base, biased):
    print(f'{token:12} {"green" if token in greens else "red":5}  base={p0:.3f}  marked={p1:.3f}')

green token의 확률은 올라가지만 red token의 확률이 0이 되지 않는다. 따라서 한 단어가 아니라 충분히 긴 sequence의 누적 count를 본다.

In [ ]:
def generate(key: bytes, length: int, delta: float, seed: int) -> list[str]:
    rng = random.Random(seed)
    tokens: list[str] = []
    for _ in range(length):
        prefix = tuple(tokens[-2:])
        greens = green_set(key, prefix)
        logits = [value + (delta if token in greens else 0.0) for token, value in zip(VOCAB, BASE_LOGITS)]
        tokens.append(rng.choices(VOCAB, weights=softmax(logits), k=1)[0])
    return tokens

def detect(tokens: list[str], key: bytes, gamma: float = GAMMA) -> tuple[int, float]:
    green_count = 0
    for index, token in enumerate(tokens):
        prefix = tuple(tokens[max(0, index - 2):index])
        green_count += token in green_set(key, prefix, gamma)
    total = len(tokens)
    z = (green_count - gamma * total) / math.sqrt(total * gamma * (1 - gamma))
    return green_count, z

key = b'classroom-key'
plain = generate(key, length=1500, delta=0.0, seed=7)
marked = generate(key, length=1500, delta=1.2, seed=7)
for label, sequence, detector_key in [
    ('plain / right key', plain, key),
    ('marked / right key', marked, key),
    ('marked / wrong key', marked, b'wrong-key-3'),
]:
    count, z = detect(sequence, detector_key)
    print(f'{label:20} green={count:4}/{len(sequence)} ({count/len(sequence):.1%}), z={z:.2f}')

## 해석

- `plain/right key`와 `marked/wrong key`는 대체로 chance 근처다.
- `marked/right key`만 큰 양의 z-score를 보인다.
- seed, vocabulary, base logit, gamma, delta를 바꾸어 quality–detectability trade-off를 관찰한다.
- 실제 detector는 repeated n-gram, tokenizer와 calibration을 더 정교하게 처리한다.